# 01 Get expected data for pileups

In [1]:
import bioframe as bf
import cooler
import cooltools
from tqdm.auto import tqdm

In [2]:
RESOLUTIONS = (10_000, 20_000)
CORES = 8
CHUNKSIZE = 1_000_000

In [3]:
clr_paths = {condition: {sis: f"/groups/gerlich/experiments/Experiments_006500/006575/coolers_repo/{condition}/{condition}.{sis}.mcool"
                         for sis in ('cis', 'trans', 'all')}
              for condition in ('WT_G2', 'dNIPBL_G2', 'dWAPL_G2', 'dCTCF_G2')}

In [4]:
clr_paths

{'WT_G2': {'cis': '/groups/gerlich/experiments/Experiments_006500/006575/coolers_repo/WT_G2/WT_G2.cis.mcool',
  'trans': '/groups/gerlich/experiments/Experiments_006500/006575/coolers_repo/WT_G2/WT_G2.trans.mcool',
  'all': '/groups/gerlich/experiments/Experiments_006500/006575/coolers_repo/WT_G2/WT_G2.all.mcool'},
 'dNIPBL_G2': {'cis': '/groups/gerlich/experiments/Experiments_006500/006575/coolers_repo/dNIPBL_G2/dNIPBL_G2.cis.mcool',
  'trans': '/groups/gerlich/experiments/Experiments_006500/006575/coolers_repo/dNIPBL_G2/dNIPBL_G2.trans.mcool',
  'all': '/groups/gerlich/experiments/Experiments_006500/006575/coolers_repo/dNIPBL_G2/dNIPBL_G2.all.mcool'},
 'dWAPL_G2': {'cis': '/groups/gerlich/experiments/Experiments_006500/006575/coolers_repo/dWAPL_G2/dWAPL_G2.cis.mcool',
  'trans': '/groups/gerlich/experiments/Experiments_006500/006575/coolers_repo/dWAPL_G2/dWAPL_G2.trans.mcool',
  'all': '/groups/gerlich/experiments/Experiments_006500/006575/coolers_repo/dWAPL_G2/dWAPL_G2.all.mcool'},


In [5]:
for condition, paths in clr_paths.items():
    for sis, path in paths.items():
        print(cooler.fileops.is_multires_file(path))

True
True
True
True
True
True
True
True
True
True
True
False


In [6]:
hg19_cens = bf.fetch_centromeres('hg19')

In [8]:
for cond, sis_data in tqdm(clr_paths.items()):
    if cond != 'dCTCF_G2':
        continue
    for sis, path in tqdm(sis_data.items(), desc=cond):
        if sis != 'all':
            continue
        for res in RESOLUTIONS:
            clr = cooler.Cooler(path + f"::/resolutions/{res}")
            chromsizes = clr.chroms()[:].rename(columns={'name': 'chrom'})
            arms = bf.make_chromarms(chromsizes, hg19_cens)
            expected = cooltools.expected_cis(clr,
                                              view_df=arms,
                                              nproc=CORES,
                                              chunksize=CHUNKSIZE)
            expected.to_parquet(f"expected.{cond}.{sis}.{res}.parquet")

  0%|          | 0/4 [00:00<?, ?it/s]

dCTCF_G2:   0%|          | 0/3 [00:00<?, ?it/s]

INFO:root:creating a Pool of 8 workers
INFO:root:creating a Pool of 8 workers
